# 06b · Qwen3.5-27B QLoRA 고해상도 + 선지 1칸 이동 추론 (A100, 학습 없음)

06에서 학습한 LoRA(마지막 체크포인트 step_473)를 불러와 **추론 해상도 1536 토큰**, **선지 1칸 순환 이동**으로 한 번 더 풉니다.
앙상블에서 06의 결과(1024 토큰, 0칸·2칸)와 평균해 Qwen3.5 자리에 씁니다. **06을 먼저 실행해야 합니다.**

순서: 설치 → 세션 다시 시작 → 데이터 → 모델(`재개: .../step_473` 확인) → 프롬프트 → valid → test
(① 점검, ② 속도 측정, 학습 셀은 없음)

In [ ]:
# ===== 0. 설치 (실행 후 '런타임 → 세션 다시 시작') =====
!pip install -q -U transformers accelerate peft bitsandbytes
!pip install -q flash-linear-attention        # 선형 어텐션(Gated DeltaNet) 빠른 커널
!pip uninstall -y -q torchao
!apt-get install -y -qq p7zip-full > /dev/null

In [ ]:
# ===== 1. 데이터 =====
import os, glob, shutil, subprocess, time, random, math
import numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")
SHARE_DIR = "/content/drive/MyDrive/ssafy/ai-challenge"
EXP_DIR = f"{SHARE_DIR}/experiment_data"
DATA_ROOT = "/content/data"
cnt = lambda d: len(glob.glob(f"{DATA_ROOT}/{d}/*.jpg"))
if cnt("train") < 6714 or cnt("test") < 6714:
    shutil.copy(glob.glob(f"{SHARE_DIR}/ssafy-16-2-ai*.zip")[0], "/content/data.zip")
    shutil.rmtree(DATA_ROOT, ignore_errors=True); os.makedirs(DATA_ROOT)
    subprocess.run(["7z","x","-y","-pocrisnotenough",f"-o{DATA_ROOT}","/content/data.zip"], check=True, stdout=subprocess.DEVNULL)
print("이미지 수 → train:", cnt("train"), "test:", cnt("test"))
resolve = lambda p: p if os.path.isabs(str(p)) else os.path.join(DATA_ROOT, str(p))
COLS = ["id","path","question","a","b","c","d","answer"]

valid_df = pd.read_csv(f"{SHARE_DIR}/split/valid_split_seed43.csv")[COLS]
clean = pd.concat([pd.read_csv(f"{SHARE_DIR}/split/train_clean_v2_seed43.csv"), pd.read_csv(f"{SHARE_DIR}/split/valid_clean_v2_seed43.csv")])[COLS]
train_df = clean[~clean.id.isin(set(valid_df.id))].reset_index(drop=True)
test_df = pd.read_csv(f"{DATA_ROOT}/test.csv")
for d in (valid_df, train_df, test_df): d["path"] = d["path"].map(resolve)
assert not set(train_df.id) & set(valid_df.id)

# 층화 순서: 정답별로 섞은 뒤 a,b,c,d를 번갈아 배치 → 앞에서 N개를 잘라도 정답 비율 유지
SEED = 43; rng = np.random.default_rng(SEED)
groups = {a: list(rng.permutation(train_df.index[train_df.answer == a])) for a in "abcd"}
order_all = []
while any(groups.values()):
    for a in "abcd":
        if groups[a]: order_all.append(groups[a].pop())
print("train 후보", len(train_df), "| valid", len(valid_df), "| test", len(test_df))

In [ ]:
# ===== 2. 모델 (다운로드 약 54GB, 5~10분) =====
import torch, torch.nn.functional as F
from PIL import Image
from transformers import AutoProcessor, BitsAndBytesConfig, get_cosine_schedule_with_warmup
try:
    from transformers import AutoModelForMultimodalLM as AutoVLM
except ImportError:
    from transformers import AutoModelForImageTextToText as AutoVLM
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
Image.MAX_IMAGE_PIXELS = None

MODEL_ID = "Qwen/Qwen3.5-27B"
TAG = "EXP-123_qwen3.5-27b_qlora"
OUT_DIR = f"{SHARE_DIR}/repro"; os.makedirs(OUT_DIR, exist_ok=True)
CKPT_DIR = f"{OUT_DIR}/models/{TAG}"   # 06과 같은 위치
TRAIN_MAX_PIXELS = 768*32*32      # 학습 이미지 최대 약 768 토큰 (속도)
INFER_MAX_PIXELS = 1536*32*32     # EXP-131: 추론 해상도 1024 → 1536 토큰
random.seed(SEED); torch.manual_seed(SEED)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoVLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="cuda", dtype=torch.bfloat16, attn_implementation="sdpa")
processor = AutoProcessor.from_pretrained(MODEL_ID)
tok = processor.tokenizer
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|out_proj|in_proj_qkvz|gate_proj|up_proj|down_proj)")
START_STEP = 0
latest = sorted([p for p in glob.glob(f"{CKPT_DIR}/step_*") if os.path.exists(f"{p}/adapter_model.safetensors")], key=lambda p: int(p.split("_")[-1]))
if latest:
    model = PeftModel.from_pretrained(model, latest[-1], is_trainable=True); START_STEP = int(latest[-1].split("_")[-1]); print("재개:", latest[-1])
else:
    model = get_peft_model(model, lora)
model.print_trainable_parameters()
n_lora = sum(1 for n, m in model.named_modules() if hasattr(m, "lora_A"))
print("LoRA 모듈 수:", n_lora, "| GPU 사용", round(torch.cuda.memory_allocated()/1e9, 1), "GB"); assert n_lora > 0

In [ ]:
# ===== 3. 프롬프트 · 선지 섞기 · 정답 토큰 loss (EXP-102와 동일, thinking 끔) =====
def fit(img, maxp):
    w, h = img.size
    if w*h > maxp:
        s = (maxp/(w*h))**0.5; img = img.resize((max(32,int(w*s)), max(32,int(h*s))), Image.BICUBIC)
    return img

PROMPT_REPEAT = 2      # 같은 질문·선지를 2번 반복 (Prompt Repetition 논문, arXiv 2512.14982)
SHIFTS = [1]           # EXP-131: EXP-123(0칸·2칸)과 겹치지 않는 1칸 이동으로 한 번만

def prompt(q, opts):
    one = (f"이미지를 자세히 보고 질문에 답하세요. 이미지 속 글자를 정확히 읽는 것이 중요합니다.\n\n"
           f"질문: {q}\n" + "".join(f"{l}. {o}\n" for l, o in zip("abcd", opts)) +
           "\n정답 선지의 알파벳(a, b, c, d) 한 글자만 출력하세요.")
    return "\n\n".join([one] * PROMPT_REPEAT)

def chat(img, text):
    msgs = [{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":text}]}]
    return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def build_train_example(r, shuffle=True):
    opts = [str(r[c]) for c in "abcd"]; gold = "abcd".index(str(r["answer"]).strip().lower())
    order = list(range(4))
    if shuffle: random.shuffle(order)
    new_opts = [opts[i] for i in order]; new_gold = "abcd"[order.index(gold)]
    img = fit(Image.open(r["path"]).convert("RGB"), TRAIN_MAX_PIXELS)
    target = new_gold + "<|im_end|>"
    enc = processor(text=[chat(img, prompt(r["question"], new_opts)) + target], images=[img], return_tensors="pt")
    n_tgt = len(tok.encode(target, add_special_tokens=False))
    labels = torch.full_like(enc["input_ids"], -100); labels[0, -n_tgt:] = enc["input_ids"][0, -n_tgt:]
    enc["labels"] = labels
    return enc

CHOICE_IDS = []
for ch in "abcd":
    ss = set()
    for x in [ch, ch.upper(), " "+ch, " "+ch.upper()]:
        t = tok.encode(x, add_special_tokens=False)
        if len(t) == 1: ss.add(t[0])
    CHOICE_IDS.append(sorted(ss))

@torch.inference_mode()
def score_once(img, q, opts):
    inp = processor(text=[chat(img, prompt(q, opts))], images=[img], return_tensors="pt").to("cuda")
    with torch.autocast("cuda", dtype=torch.bfloat16):
        try: lg = model(**inp, logits_to_keep=1, use_cache=False).logits[0, -1].float()
        except TypeError: lg = model(**inp, use_cache=False).logits[0, -1].float()
    return F.softmax(torch.stack([torch.logsumexp(lg[i], 0) for i in CHOICE_IDS]), 0).cpu().numpy()

def score(r):
    img = fit(Image.open(r["path"]).convert("RGB"), INFER_MAX_PIXELS)
    orig = [str(r[c]) for c in "abcd"]; acc = np.zeros(4)
    for k in SHIFTS:
        order = [(j + k) % 4 for j in range(4)]             # 표시 위치 j에 원래 선지 order[j]
        p = score_once(img, r["question"], [orig[i] for i in order])
        for j, i in enumerate(order): acc[i] += p[j]
    return acc / len(SHIFTS)

In [ ]:
# ===== valid 672 =====
model.eval(); P = ["prob_a","prob_b","prob_c","prob_d"]
OUT_TAG = "qwen35_px1536"
def run(df, out_csv, every=50):
    done = pd.read_csv(out_csv) if os.path.exists(out_csv) else pd.DataFrame(columns=["id"]+P)
    todo = df[~df.id.astype(str).isin(set(done.id.astype(str)))]; rows = []; t0 = time.time()
    print(f"완료 {len(done)} / 남음 {len(todo)}")
    for n, (_, r) in enumerate(todo.iterrows(), 1):
        try: p = score(r)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); p = np.full(4, .25); print("OOM", r.id)
        rows.append([r.id, *p])
        if n % every == 0 or n == len(todo):
            done = pd.concat([done, pd.DataFrame(rows, columns=done.columns)], ignore_index=True); done.to_csv(out_csv, index=False); rows = []
            el = time.time()-t0; print(f"{n}/{len(todo)} {el/n:.2f}s/문항, 남은 {el/n*(len(todo)-n)/60:.0f}분")
    return done.set_index("id")
vp = run(valid_df, f"{OUT_DIR}/{OUT_TAG}_valid_probs.csv")
ids = valid_df.id.values; ans = valid_df.answer.map(dict(zip("abcd", range(4)))).values
q1 = vp.loc[ids, P].values.astype(float)
q0 = pd.read_csv(f"{OUT_DIR}/qwen35_valid_probs.csv").set_index("id").loc[ids, P].values.astype(float)
ok = lambda p: (p.argmax(1) == ans).sum()
print(f"06 {ok(q0)} | 06b {ok(q1)} | 평균 {ok((q0+q1)/2)} | 2:1 평균 {ok((2*q0+q1)/3)} /672 | 답이 다른 문항 {(q0.argmax(1)!=q1.argmax(1)).sum()}")

## test 6,714 (끊기면 이 셀만 다시 실행)

In [ ]:
tp = run(test_df, f"{OUT_DIR}/{OUT_TAG}_test_probs.csv", every=100)
print(tp.shape)